In [1]:
# This test program aims to output a list of n_mc coordinates within a sample.
# This program will use ray-tracing and the odd-even rule to determine whether a random coordinate is within a sample.

using MeshIO
using FileIO
using GeometryBasics
using Distributions
using StaticArrays
using LinearAlgebra
using BenchmarkTools

In [2]:
# Setting the desired number of MC sample points.

const n_mc = 1000

1000

In [3]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.

# Dummy multiple-crystal sample comprising 7 icospheres with 80 faces each.
stl = load("STL_FileExamples/7_Icospheres80.stl")
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const n_faces = length(indices)

560

In [4]:
# Finding the maximum and minimum value of each coordinate.

max_coord = [maximum(getindex.(vertices, 1)), maximum(getindex.(vertices, 2)), maximum(getindex.(vertices, 3))]
min_coord = [minimum(getindex.(vertices, 1)), minimum(getindex.(vertices, 2)), minimum(getindex.(vertices, 3))]

3-element Vector{Float32}:
 -3.9510565
 -4.0
 -4.0

In [5]:
# Storing the coordinates into a matrix of length n_mc.

mc_coords = Float32.(zeros(n_mc, 3))

1000×3 Matrix{Float32}:
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 ⋮         
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0
 0.0  0.0  0.0

In [6]:
# Calculating and storing the vectors parallel to each face, e2 = V2 - V1 and e3 = V3 - V1, and V1s specifically in preparation for the Moller-Trumbore Algorithm.
# The vertices of the triangular faces are labelled V1, V2, V3.

v1s = vertices[getindex.(indices, 1)]
e2s = vertices[getindex.(indices, 2)] - vertices[getindex.(indices, 1)]
e3s = vertices[getindex.(indices, 3)] - vertices[getindex.(indices, 1)]
# Converting the 3-vectors within e2s and e3s to static arrays.
v1s = [SVector{3, Float32}(vec[1], vec[2] ,vec[3]) for vec in v1s]
e2s = [SVector{3, Float32}(vec[1], vec[2] ,vec[3]) for vec in e2s]
e3s = [SVector{3, Float32}(vec[1], vec[2] ,vec[3]) for vec in e3s]

560-element Vector{SVector{3, Float32}}:
 [0.31139204, 0.041633785, -0.4472136]
 [-0.2763932, 0.14934921, -0.4472136]
 [-0.31139204, 0.041633785, 0.4472136]
 [0.22744972, 0.21671408, 0.4472136]
 [-0.31139204, -0.041633785, 0.4472136]
 [-0.29828143, -0.21671408, -0.4034372]
 [0.29828143, 0.21671408, 0.4034372]
 [0.41179773, 0.3506508, 0.0785175]
 [0.5257311, 0.0, -0.14934921]
 [0.41179773, -0.3506508, 0.0785175]
 ⋮
 [-0.16245985, 0.5, 0.14934921]
 [-0.41179773, -0.3506508, -0.07851744]
 [0.36327124, 0.5, 0.0]
 [-0.5257311, 0.0, 0.14934921]
 [0.20623624, -0.5, -0.07851744]
 [-0.36327124, 0.5, 0.0]
 [-0.16245985, -0.5, 0.14934921]
 [0.5392587, 0.041633785, -0.07851744]
 [-0.58778524, -0.190983, 0.0]

In [7]:
# Defining the function to determine the number of times the neutron intersects the sample.

"""
Calculates how many faces a neutron intersects given its direction vector. Accomplishes this by considering ray-triangle intersections.
Iterates through each face to determine which one is intersected.
Exploits the method described in 'Fast, Minimum Storage Ray-Triangle Intersection' by Moller and Trumbore.

Parameters
----------
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
d (3-vector with float elements): Normalised direction vector.
ps (n_faces-vector of 3-vectors with float elements): Array containing p = d x e3 for each face.
dets (n_faces-vector with float elements): Array containing det = p.e2 = (d x e3).e2 for each face.
origin (3-vector with float elements): Coordinates of scattering sites.
v1s (n_faces-vector of 3-vectors with float elements): First vertex of each face, V1.
λs (vector with float elements): Empty vector that will store distances between origin and intersection points, in units of the .stl file.
path_lengths (vector with float elements): Empty vector that will store non-duplicate distances, in units of the .stl file.

Returns
-------
n_int (integer): Number of intersections.
"""
function int_calc(
    e2s :: Vector{SVector{3, Float32}}, 
    e3s :: Vector{SVector{3, Float32}}, 
    d :: SVector{3, Float32}, 
    ps :: Vector{SVector{3, Float32}}, 
    dets :: Vector{Float32}, 
    origin :: Vector{Float32}, 
    v1s :: Vector{SVector{3, Float32}},
    λs :: Vector{Float32}, 
    path_lengths :: Vector{Float32}
    ) :: Integer
    # Emptying the pre-allocated path length stores.
    empty!(λs)
    empty!(path_lengths)
    # Iterating through all faces.
    @inbounds for j in 1:n_faces
        # If det = p.e2 = (d x e3).e2 = 0, the path is parallel to the triangular face, so it can never intersect it.
        # As we are working with multiple crystals, no culling of back- or front-facing triangles can be done.
        det = dets[j]
        if abs(det) > 1e-6
            inv_det = 1 / det
            # Calculating t = origin - V1 and q = t x e2 required for the MT algorithm.
            t = origin - v1s[j]
            q = cross(t, e2s[j])
            # Calculating the barycentric coordinates, (u,v), of the intersection.
            u = inv_det * (dot(ps[j], t))
            v = inv_det * (dot(q, d))
            # Determining whether the intersection point lies within the triangle.
            if v ≥ 0 && u ≥ 0 && (u + v) ≤ 1
                # Neutron's path described by r(λ) = origin + λd.
                λ = inv_det * dot(q, e3s[j])
                # Accepting only positive λ corresponding to forward direction.
                if λ > 0
                    # The path length is simply λ as the direction vector is normalised.
                    push!(λs, λ)
                end
            end
        end
    end
    if isempty(λs)
        # Returning 0 if no surface is intersected.
        return 0
    end
    # Ordering the path lengths.
    sort!(λs)
    # Filling the array with non-duplicate lengths in order.
    # Duplicate path lengths arise from paths near a vertex between faces.
    push!(path_lengths, λs[1])
    for i in λs
        if abs(i - last(path_lengths)) > 1e-6
            push!(path_lengths, i)
        end
    end
    # Finding the number of intersections.
    n_int = length(path_lengths)
    return n_int
end

# p = Vector{SVector{3, Float32}}(undef, n_faces)
# det = Vector{Float32}(undef, n_faces)
# d = SVector{3, Float32}([1, 0, 0])
# pdet_calc!(d, e2s, e3s, p, det)
# λs = Vector{Float32}(undef, 10)
# path_lengths = Vector{Float32}(undef, 10)
# display(int_calc(e2s, e3s, d, p, det, mc_coords[6, :], v1s, λs, path_lengths))
# display(mc_coords[6, :])

int_calc

In [8]:
# Defining a function to pre-calculate p = d x e3 and determinant = p.e2 = (d x e3).e2 required for the MT Algorithm.

"""
Calculates p = d x e3 and determinant = p.e2 = (d x e3).e2 required for the MT Algorithm.

Parameters
----------
d (3-vector with float elements): Normalised direction vector.
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
ps (n_faces-vector of 3-vectors with float elements): Pre-allocated vector.
dets (n_faces-vector with float elements): Pre-allocated vector.

Returns
-------
ps (n_faces-vector of 3-vectors with float elements): Array containing p = d x e3 for each face.
dets (n_faces-vector with float elements): Array containing det = p.e2 = (d x e3).e2 for each face.
"""
function pdet_calc!(
    d :: SVector{3, Float32}, 
    e2s :: Vector{SVector{3, Float32}}, 
    e3s :: Vector{SVector{3, Float32}},
    ps :: Vector{SVector{3, Float32}},
    dets :: Vector{Float32} 
    ) :: Tuple{Vector{SVector{3, Float32}}, Vector{Float32}}
    # @inbounds is used to remove checks on the index i as we are sure of the sizes of our arrays.
    # @simd is used to vectorize and speed up the loop.
    @inbounds @simd for i in 1:n_faces
        # Calculating cross products, p = d x e3, for the direction vector, d, and for each face.
        ps[i] = cross(d, e3s[i])
        # Calculating the determinant = p.e2 = (d x e3).e2 for the direction vector, d, and for each face.
        dets[i] = dot(ps[i], e2s[i])
    end
    return ps, dets
end


pdet_calc!

In [9]:
# Defining a function to generate the desired number of MC sample points.

"""
Generates the required number of MC sample points.

Parameters
----------
min_coord (3-vector with float elements): Minimum value of each x, y and z coordinate, in units of .stl file.
max_coord (3-vector with float elements): Maximum value of each x, y and z coordinate, in units of .stl file.
e2s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V2 - V1.
e3s (n_faces-vector of 3-vectors with float elements): Array containing vectors parallel to each face, equal to V3 - V1.
v1s (n_faces-vector of 3-vectors with float elements): First vertex of each face, V1.
mc_coords (n_mc-vector of 3-vectors with float elements): Empty matrix of coordinates of sample points.

Returns
-------
mc_coords (n_mc-vector of 3-vectors with float elements): Coordinates of sample points used in MC method.
"""

function sampling!(
    min_coord :: Vector{Float32}, 
    max_coord :: Vector{Float32}, 
    e2s :: Vector{SVector{3, Float32}}, 
    e3s :: Vector{SVector{3, Float32}}, 
    v1s :: Vector{SVector{3, Float32}}, 
    mc_coords :: Matrix{Float32}
    ) :: Matrix{Float32}
    # Pre-allocating the necessary vectors.
    λs = Vector{Float32}(undef, 10)
    path_lengths = Vector{Float32}(undef, 10)
    p = Vector{SVector{3, Float32}}(undef, n_faces)
    det = Vector{Float32}(undef, n_faces)
    test = Vector{Float32}(undef, 3)
    # Pre-calculating uniform distribution describing the volume our crystal(s) lies in.
    x_range = Uniform(min_coord[1], max_coord[1])
    y_range = Uniform(min_coord[2], max_coord[2])
    z_range = Uniform(min_coord[3], max_coord[3])
    # Sending a dummy neutron along the x direction, starting at this test coordinate.
    d = SVector{3, Float32}([1, 0, 0])
    # Calculating p and det required for the MT algorithm.
    pdet_calc!(d, e2s, e3s, p, det)
    # Tallying the number of accepted coordinates.
    n_acc = 0
    # Continuing this sample generation until there are n_mc coordinates inside the crytal(s).
    while n_acc < n_mc
        # Generating a random coordinate within the pre-defined sample range.
        x = rand(x_range)
        y = rand(y_range)
        z = rand(z_range)
        test[1] = x
        test[2] = y
        test[3] = z
        # Calculating the number of intersections this theoretical neutron makes with the sample surfaces.
        n_int = int_calc(e2s, e3s, d, p, det, test, v1s, λs, path_lengths)
        # If it makes an even number of intersections, it is outside a sample.
        # Odd number of intersections means it began in a sample.
        if isodd(n_int)
            # Accepting this test coordinate.
            mc_coords[n_acc + 1, :] = test
            n_acc += 1
        end
    end
    return mc_coords
end

sampling! (generic function with 1 method)

In [11]:
# Benchmarking this sample point generation function.

@benchmark sampling!(min_coord, max_coord, e2s, e3s, v1s, mc_coords)

BenchmarkTools.Trial: 34 samples with 1 evaluation per sample.
 Range (min … max):  121.963 ms … 176.619 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     151.294 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   150.160 ms ±  16.027 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

    ▃              █   ▃      ▃   ▃         ▃▃▃            ▃     
  ▇▁█▁▇▁▁▁▁▇▇▁▇▁▁▁▁█▁▁▁█▁▁▇▁▁▁█▇▁▁█▇▁▁▇▇▁▁▁▁███▁▁▁▁▁▇▇▇▁▁▇▁█▁▁▇ ▁
  122 ms           Histogram: frequency by time          177 ms <

 Memory estimate: 9.20 KiB, allocs estimate: 13.

In [12]:
sampling!(min_coord, max_coord, e2s, e3s, v1s, mc_coords)
mc_coords

1000×3 Matrix{Float32}:
 -0.502336    0.406969    -2.62923
 -0.368813   -0.661392     3.31235
  0.590703    2.91006     -0.151619
  0.118745   -0.43968     -2.38949
 -0.579074    3.07065     -0.357333
 -0.140454    2.16798     -0.290406
 -0.0938784   0.927691    -2.91937
 -0.281817    2.99979      0.773581
 -0.649253    0.491187     3.28852
  0.539579    3.35202      0.526109
  ⋮                       
 -0.44026    -0.607659     0.409723
 -0.154568   -3.21657     -0.508984
 -0.75153    -3.20398     -0.133196
 -0.237759    0.0613468    2.58952
  0.688857   -2.96223     -0.0121565
 -0.242752   -0.646867    -0.0250608
 -0.0141331  -2.34901     -0.679076
 -0.357464   -0.00956564   0.629199
 -0.742814   -3.17932      0.0365196